# GAM-Style Additive Spline Baseline

## tl;dr

This notebook tests generalized-additive-model style spline regressions before moving to LightGBM/XGBoost. The implementation uses sklearn `SplineTransformer` plus Ridge regularization so the same logic can be copied into a self-contained Kaggle notebook without extra dependencies.

## Context & Methods

Phase 2-4 EDA showed the problem is dominated by trailing-zone extrapolation, endpoint hold is a strong baseline, and direct recent-slope extrapolation is unstable. This notebook predicts target-zone `delta_tvt_from_boundary`, then reconstructs `TVT = last_known_tvt + predicted_delta`.

### Key Assumptions

- Feature construction is notebook-local and Kaggle-safe; no precomputed artifacts are needed for model fitting.
- Validation uses grouped well folds and sampled target-zone rows from train wells. This is a target-zone diagnostic protocol, not yet a Kaggle submission notebook.
- Spline terms are additive. If performance is weak, that suggests boosted-tree interaction discovery may matter.


In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import math
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well, load_typewell
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "gam_spline_baseline"
RUN_DATE = "2026-06-21"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 80000
SPLINE_KNOTS = 7
SPLINE_DEGREE = 3
RIDGE_ALPHA = 30.0

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"
MODEL_DIR = ARTIFACT_DIR / "models" / "gam"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")


In [2]:
"Define inline feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def clean_gr(series):
    gr = safe_numeric(series).clip(lower=0, upper=300)
    return gr.fillna(gr.rolling(21, center=True, min_periods=1).median()).ffill().bfill()


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def typewell_interpolator(typewell):
    if typewell.empty or "TVT" not in typewell.columns or "GR" not in typewell.columns:
        return None
    tw = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(tw) < 2:
        return None
    return interp1d(
        tw["TVT"].to_numpy(float),
        tw["GR"].to_numpy(float),
        bounds_error=False,
        fill_value=(float(tw["GR"].iloc[0]), float(tw["GR"].iloc[-1])),
    )


def known_typewell_corr(horizontal, typewell):
    interp = typewell_interpolator(typewell)
    if interp is None:
        return np.nan
    known = horizontal["TVT_input"].notna() & horizontal["GR"].notna()
    if int(known.sum()) < 20:
        return np.nan
    sample_pos = deterministic_sample_positions(known, 500)
    sample = horizontal.iloc[sample_pos]
    tw_gr = interp(sample["TVT_input"].to_numpy(float))
    if np.nanstd(tw_gr) < 1e-9 or np.nanstd(sample["GR"].to_numpy(float)) < 1e-9:
        return np.nan
    return float(np.corrcoef(sample["GR"].to_numpy(float), tw_gr)[0, 1])


def build_target_zone_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    typewell_path = directory / f"{well}__typewell.csv"
    typewell = load_typewell(typewell_path) if typewell_path.exists() else pd.DataFrame()

    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])
    gr_raw = safe_numeric(horizontal["GR"])
    gr_imputed = clean_gr(horizontal["GR"])
    gr_missing = gr_raw.isna().astype(float)

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])
    last_known_gr = float(gr_imputed.iloc[last_known_pos])
    last_known_x = float(x.iloc[last_known_pos])
    last_known_y = float(y.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{w}": window_slope(md, tvt_input, last_known_pos, w) for w in slope_windows}
    z_slopes = {f"z_slope_last_{w}": window_slope(md, z, last_known_pos, w) for w in slope_windows}

    tw_interp = typewell_interpolator(typewell)
    typewell_corr = known_typewell_corr(horizontal, typewell)
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()
    gr_roll_mean_25 = gr_imputed.rolling(25, center=True, min_periods=1).mean()
    gr_roll_std_25 = gr_imputed.rolling(25, center=True, min_periods=2).std()
    gr_missing_frac_25 = gr_missing.rolling(25, center=True, min_periods=1).mean()
    gr_missing_frac_100 = gr_missing.rolling(100, center=True, min_periods=1).mean()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)
    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        slope50 = tvt_slopes["tvt_slope_last_50"]
        slope50_delta = 0.0 if not np.isfinite(slope50) else slope50 * distance_md
        tvt_proxy = last_known_tvt + slope50_delta
        tw_gr_proxy = float(tw_interp([tvt_proxy])[0]) if tw_interp is not None and np.isfinite(tvt_proxy) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "target_rows_in_well": int(target_mask.sum()),
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "last_known_md": last_known_md,
            "last_known_z": last_known_z,
            "last_known_gr": last_known_gr,
            "dX_from_boundary": float(x.iloc[pos] - last_known_x),
            "dY_from_boundary": float(y.iloc[pos] - last_known_y),
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "GR_imputed": float(gr_imputed.iloc[pos]),
            "GR_missing": float(gr_missing.iloc[pos]),
            "gr_roll_mean_25": float(gr_roll_mean_25.iloc[pos]),
            "gr_roll_std_25": float(gr_roll_std_25.iloc[pos]) if np.isfinite(gr_roll_std_25.iloc[pos]) else np.nan,
            "gr_missing_frac_25": float(gr_missing_frac_25.iloc[pos]),
            "gr_missing_frac_100": float(gr_missing_frac_100.iloc[pos]),
            "tw_gr_at_proxy": tw_gr_proxy,
            "gr_minus_tw_proxy": float(gr_imputed.iloc[pos] - tw_gr_proxy) if np.isfinite(tw_gr_proxy) else np.nan,
            "typewell_gr_corr": typewell_corr,
            "TVT": float(horizontal["TVT"].iloc[pos]),
        }
        for key, value in tvt_slopes.items():
            row[key] = value
            row[f"{key}_projected_delta"] = value * distance_md if np.isfinite(value) else np.nan
        for key, value in z_slopes.items():
            row[key] = value
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        row["endpoint_hold_pred"] = last_known_tvt
        row["slope50_pred"] = last_known_tvt + slope50_delta
        rows.append(row)
    return pd.DataFrame(rows)


In [3]:
"Build the sampled target-zone modeling frame."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_target_zone_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

print({
    "wells": len(train_wells),
    "sample_rows": len(feature_frame),
    "build_seconds": round(build_seconds, 2),
})
feature_frame.head()


20:05:46 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 117.96}


,wellname,row_idx,target_rows_in_well,MD,Z,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,last_known_md,...,tvt_slope_last_50_projected_delta,tvt_slope_last_100,tvt_slope_last_100_projected_delta,z_slope_last_10,z_slope_last_25,z_slope_last_50,z_slope_last_100,delta_tvt_from_boundary,endpoint_hold_pred,slope50_pred
0,000d7d20,1442,3836,12909.0,-9735.07,0.273261,1.0,0.000000,11747.37,12908.0,...,0.016327,0.008485,0.008485,0.007778,0.005833,0.002653,0.004343,0.01,11747.37,11747.386327
1,000d7d20,1469,3836,12936.0,-9734.69,0.278378,28.0,0.007040,11747.37,12908.0,...,0.457143,0.008485,0.237576,0.007778,0.005833,0.002653,0.004343,0.15,11747.37,11747.827143
2,000d7d20,1497,3836,12964.0,-9734.06,0.283684,56.0,0.014342,11747.37,12908.0,...,0.914286,0.008485,0.475152,0.007778,0.005833,0.002653,0.004343,0.09,11747.37,11748.284286
3,000d7d20,1524,3836,12991.0,-9733.28,0.288800,83.0,0.021382,11747.37,12908.0,...,1.355102,0.008485,0.704242,0.007778,0.005833,0.002653,0.004343,0.68,11747.37,11748.725102
4,000d7d20,1552,3836,13019.0,-9732.40,0.294106,111.0,0.028683,11747.37,12908.0,...,1.812245,0.008485,0.941818,0.007778,0.005833,0.002653,0.004343,1.29,11747.37,11749.182245


In [4]:
"Define GAM-style additive spline variants."

endpoint_features = ["last_known_tvt", "distance_from_boundary_md", "eval_progress"]
trajectory_features = ["MD", "Z", "md_norm", "dX_from_boundary", "dY_from_boundary", "dZ_from_boundary", "horizontal_dist_from_boundary"]
slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
gr_features = ["GR_imputed", "GR_missing", "last_known_gr", "gr_roll_mean_25", "gr_roll_std_25", "gr_missing_frac_25", "gr_missing_frac_100"]
typewell_features = ["tw_gr_at_proxy", "gr_minus_tw_proxy", "typewell_gr_corr"]

variant_specs = [
    {"name": "endpoint_boundary", "smooth": ["distance_from_boundary_md", "eval_progress"], "linear": ["last_known_tvt"]},
    {"name": "boundary_trajectory", "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"], "linear": ["last_known_tvt", "last_known_z"]},
    {"name": "boundary_trajectory_gr", "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary", "GR_imputed", "gr_roll_mean_25", "gr_missing_frac_25"], "linear": ["last_known_tvt", "last_known_gr", "GR_missing"]},
    {"name": "boundary_trajectory_slopes", "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features, "linear": ["last_known_tvt"]},
    {"name": "full_candidate", "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary", "GR_imputed", "gr_roll_mean_25", "gr_missing_frac_25"] + slope_features + slope_projection_features + ["tw_gr_at_proxy", "gr_minus_tw_proxy"], "linear": ["last_known_tvt", "last_known_gr", "GR_missing", "typewell_gr_corr"]},
]
variant_specs = [
    {**spec, "features": list(dict.fromkeys(spec["smooth"] + spec["linear"]))}
    for spec in variant_specs
]
variant_table = pd.DataFrame([
    {"variant": spec["name"], "n_smooth_features": len(spec["smooth"]), "n_linear_features": len(spec["linear"]), "n_total_features": len(spec["features"])}
    for spec in variant_specs
])
variant_table


,variant,n_smooth_features,n_linear_features,n_total_features
0,endpoint_boundary,2,1,3
1,boundary_trajectory,5,2,7
2,boundary_trajectory_gr,8,3,11
3,boundary_trajectory_slopes,16,1,17
4,full_candidate,21,4,25


In [5]:
"Define additive spline model fitting and validation helpers."


def make_additive_spline_model(smooth_features, linear_features):
    transformers = []
    if smooth_features:
        transformers.append((
            "smooth",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("splines", SplineTransformer(n_knots=SPLINE_KNOTS, degree=SPLINE_DEGREE, include_bias=False)),
            ]),
            smooth_features,
        ))
    if linear_features:
        transformers.append((
            "linear",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            linear_features,
        ))
    return Pipeline([
        ("features", ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA)),
    ])


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def metric_row(y_true, pred_tvt):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))),
        "mae": float(mean_absolute_error(y_true, pred_tvt)),
        "r2": float(r2_score(y_true, pred_tvt)),
    }


def evaluate_endpoint(valid_frame):
    return valid_frame["endpoint_hold_pred"].to_numpy(float)


def evaluate_slope50(valid_frame):
    return valid_frame["slope50_pred"].to_numpy(float)


In [6]:
"Run grouped well-fold validation for GAM-style variants and baseline references."

fold_rows = []
residual_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)
for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true = valid_frame["TVT"].to_numpy(float)

    for baseline_name, pred in [("endpoint_hold", evaluate_endpoint(valid_frame)), ("slope50_projection", evaluate_slope50(valid_frame))]:
        metrics = metric_row(y_true, pred)
        fold_rows.append({"fold": fold_idx, "model_name": baseline_name, "model_family": "baseline", "n_train_rows": 0, "n_valid_rows": len(valid_frame), **metrics})
        if baseline_name == "endpoint_hold":
            sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), 5000)).astype(int)
            residual_rows.append(pd.DataFrame({
                "fold": fold_idx,
                "model_name": baseline_name,
                "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(),
                "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(),
                "y_true": y_true[sample_idx],
                "y_pred": pred[sample_idx],
                "residual": y_true[sample_idx] - pred[sample_idx],
            }))

    for spec in variant_specs:
        model = make_additive_spline_model(spec["smooth"], spec["linear"])
        model.fit(train_frame[spec["features"]], train_frame["delta_tvt_from_boundary"])
        pred_delta = model.predict(valid_frame[spec["features"]])
        pred_tvt = valid_frame["last_known_tvt"].to_numpy(float) + pred_delta
        metrics = metric_row(y_true, pred_tvt)
        fold_rows.append({"fold": fold_idx, "model_name": spec["name"], "model_family": "gam_spline", "n_train_rows": len(train_frame), "n_valid_rows": len(valid_frame), **metrics})
        if spec["name"] in {"boundary_trajectory", "full_candidate"}:
            sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), 5000)).astype(int)
            residual_rows.append(pd.DataFrame({
                "fold": fold_idx,
                "model_name": spec["name"],
                "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(),
                "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(),
                "y_true": y_true[sample_idx],
                "y_pred": pred_tvt[sample_idx],
                "residual": y_true[sample_idx] - pred_tvt[sample_idx],
            }))

fold_metrics = pd.DataFrame(fold_rows)
summary = (
    fold_metrics.groupby(["model_family", "model_name"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"))
    .sort_values("rmse_mean")
)
residual_sample = pd.concat(residual_rows, ignore_index=True)
summary


,model_family,model_name,rmse_mean,rmse_std,mae_mean,r2_mean,n_valid_rows,n_train_rows_mean
4,gam_spline,boundary_trajectory_slopes,14.871892,0.533661,10.489798,0.999423,108220,72146.666667
6,gam_spline,full_candidate,14.935106,0.541216,10.565473,0.999418,108220,72146.666667
2,gam_spline,boundary_trajectory,15.371539,0.450851,10.870071,0.999384,108220,72146.666667
3,gam_spline,boundary_trajectory_gr,15.387913,0.429427,10.898906,0.999382,108220,72146.666667
5,gam_spline,endpoint_boundary,15.524424,0.581619,10.867098,0.999371,108220,72146.666667
0,baseline,endpoint_hold,15.578175,0.651555,10.905266,0.999367,108220,0.000000
1,baseline,slope50_projection,104.615744,14.849336,51.109967,0.971162,108220,0.000000


In [7]:
"Export tables and diagnostic figures."

run_hint = RUN_TAG
fold_metrics_path = TABLE_DIR / f"model__gam_spline_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__gam_spline_summary__{RUN_DATE}.csv"
variant_path = TABLE_DIR / f"model__gam_spline_variants__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__gam_spline_residual_sample__{RUN_DATE}.csv"
fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
variant_table.to_csv(variant_path, index=False)
residual_sample.to_csv(residual_path, index=False)

fig_paths = {}
plot_summary = summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4C78A8" if fam == "gam_spline" else "#F58518" for fam in plot_summary["model_family"]]
ax.barh(plot_summary["model_name"], plot_summary["rmse_mean"], color=colors)
ax.set_title("GAM-Style Spline Variant RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__gam_spline_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for model_name, group in fold_metrics.groupby("model_name"):
    if model_name in ["endpoint_hold", "slope50_projection", "boundary_trajectory", "full_candidate", summary.iloc[0]["model_name"]]:
        ax.plot(group["fold"], group["rmse"], marker="o", label=model_name)
ax.set_title("GAM-Style Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend(fontsize=8)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__gam_spline_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

best_name = str(summary.iloc[0]["model_name"])
best_residuals = residual_sample.loc[residual_sample["model_name"] == best_name]
if best_residuals.empty:
    best_residuals = residual_sample.loc[residual_sample["model_name"] == "full_candidate"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["y_pred"], best_residuals["residual"], s=6, alpha=0.35, color="#54A24B")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residuals for {best_name}")
ax.set_xlabel("predicted TVT")
ax.set_ylabel("true - predicted")
fig.tight_layout()
fig_paths["best_residuals"] = FIGURE_DIR / f"model__gam_spline_best_residuals__{RUN_DATE}.png"
fig.savefig(fig_paths["best_residuals"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
boundary_model = "boundary_trajectory" if "boundary_trajectory" in residual_sample["model_name"].unique() else best_name
scatter_frame = residual_sample.loc[residual_sample["model_name"] == boundary_model].merge(
    feature_frame[["wellname", "row_idx", "distance_from_boundary_md"]],
    on=["wellname", "row_idx"],
    how="left",
)
ax.hexbin(scatter_frame["distance_from_boundary_md"], scatter_frame["residual"], gridsize=45, mincnt=1, cmap="viridis")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"{boundary_model} Residuals vs Boundary Distance")
ax.set_xlabel("MD distance past boundary")
ax.set_ylabel("true - predicted")
fig.tight_layout()
fig_paths["residual_by_boundary"] = FIGURE_DIR / f"model__gam_spline_residual_by_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["residual_by_boundary"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}


{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_rmse_comparison__2026-06-21.png',
 'fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_fold_rmse__2026-06-21.png',
 'best_residuals': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_best_residuals__2026-06-21.png',
 'residual_by_boundary': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_residual_by_boundary__2026-06-21.png'}

In [8]:
"Write report, update comparison table, and create run bundle."

best = summary.iloc[0]
best_gam = summary.loc[summary["model_family"] == "gam_spline"].iloc[0]
endpoint = summary.loc[summary["model_name"] == "endpoint_hold"].iloc[0]
slope50 = summary.loc[summary["model_name"] == "slope50_projection"].iloc[0]
rf_reference_rmse = 16.06550629767113
ridge_reference_rmse = 15.765977474514608
phase4_endpoint_rmse = 15.909852870734554

findings = pd.DataFrame([
    {
        "topic": "best_gam_variant",
        "finding": f"Best GAM-style variant is {best_gam.model_name} with sampled grouped RMSE {best_gam.rmse_mean:.4f}.",
        "modeling_implication": "Use this as the first additive-smooth baseline before boosted-tree tuning.",
    },
    {
        "topic": "endpoint_comparison",
        "finding": f"Best GAM-style RMSE {best_gam.rmse_mean:.4f} vs sampled endpoint-hold RMSE {endpoint.rmse_mean:.4f}.",
        "modeling_implication": "A useful model must clear endpoint hold under the active validation protocol.",
    },
    {
        "topic": "slope_projection_check",
        "finding": f"Direct last-50 slope projection remains unstable with sampled RMSE {slope50.rmse_mean:.4f}.",
        "modeling_implication": "Slope should remain a feature, not a standalone extrapolator.",
    },
    {
        "topic": "historical_reference",
        "finding": f"Reference local CV: Ridge {ridge_reference_rmse:.4f}, Random Forest {rf_reference_rmse:.4f}; this GAM run uses sampled target-zone CV, so compare directionally, not as a final leaderboard proxy.",
        "modeling_implication": "If GAM is competitive, create a self-contained Kaggle notebook and rerun against the hidden-test workflow.",
    },
])
findings_path = TABLE_DIR / f"model__gam_spline_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"model__gam_spline_baseline__{RUN_DATE}.md"
report_lines = [
    "# GAM-Style Additive Spline Baseline",
    "",
    f"Date: {RUN_DATE}",
    "",
    "## Key Findings",
]
for row in findings.itertuples(index=False):
    report_lines.extend([
        f"- **{row.topic}**: {row.finding}",
        f"  - Modeling implication: {row.modeling_implication}",
    ])
report_lines.extend([
    "",
    "## Artifacts",
    f"- Fold metrics: {fold_metrics_path}",
    f"- Summary: {summary_path}",
    f"- Variant table: {variant_path}",
    f"- Residual sample: {residual_path}",
    f"- Findings: {findings_path}",
])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{
    "run_id": "pending_run_bundle",
    "model_family": "gam_spline",
    "model_name": str(best_gam.model_name),
    "cv_rmse_mean": float(best_gam.rmse_mean),
    "cv_rmse_std": float(best_gam.rmse_std),
    "train_time_s": float(build_seconds),
    "n_features": int(variant_table.loc[variant_table["variant"] == best_gam.model_name, "n_total_features"].iloc[0]),
    "notes": "GAM-style additive spline Ridge; sampled target-zone grouped CV; notebook-local Kaggle-safe features",
}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {
    "task": "gam_spline_baseline",
    "seed": SEED,
    "n_splits": N_SPLITS,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "spline_knots": SPLINE_KNOTS,
    "spline_degree": SPLINE_DEGREE,
    "ridge_alpha": RIDGE_ALPHA,
    "note": "Notebook-local GAM-style additive spline regression; target-zone diagnostic CV.",
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("best_gam_rmse", float(best_gam.rmse_mean))
    rb.add_metric("best_gam_rmse_std", float(best_gam.rmse_std))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("slope50_projection_rmse", float(slope50.rmse_mean))
    rb.add_metric("best_gam_minus_endpoint", float(best_gam.rmse_mean - endpoint.rmse_mean))
    for output_path in [fold_metrics_path, summary_path, variant_path, residual_path, findings_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("GAM-style spline baseline completed. Feature construction is notebook-local for later Kaggle submission portability.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({
    "run_id": run_id,
    "best_gam": best_gam.model_name,
    "best_gam_rmse": round(float(best_gam.rmse_mean), 4),
    "endpoint_hold_rmse": round(float(endpoint.rmse_mean), 4),
    "best_minus_endpoint": round(float(best_gam.rmse_mean - endpoint.rmse_mean), 4),
})
findings


{'run_id': '2026-06-21__2007__gam_spline_baseline__8046b3f', 'best_gam': 'boundary_trajectory_slopes', 'best_gam_rmse': 14.8719, 'endpoint_hold_rmse': 15.5782, 'best_minus_endpoint': -0.7063}


,topic,finding,modeling_implication
0,best_gam_variant,Best GAM-style variant is boundary_trajectory_...,Use this as the first additive-smooth baseline...
1,endpoint_comparison,Best GAM-style RMSE 14.8719 vs sampled endpoin...,A useful model must clear endpoint hold under ...
2,slope_projection_check,Direct last-50 slope projection remains unstab...,"Slope should remain a feature, not a standalon..."
3,historical_reference,"Reference local CV: Ridge 15.7660, Random Fore...","If GAM is competitive, create a self-contained..."


## Results

The executed notebook exports variant metrics, residual diagnostics, a report, updates the model comparison table, and writes a run bundle. The result is a GAM-style target-zone diagnostic, not yet a Kaggle submission.

## Takeaways

If the additive spline model beats endpoint hold and is near or better than prior baselines, the next step is a self-contained Kaggle notebook or a richer LightGBM/XGBoost comparison using the same notebook-local feature logic.
